# Lab 4 — Section B: Sum of Prime Numbers

**Task:** Find the sum of every prime number strictly smaller than a given limit.

This section compares three approaches:
1. Trial division using every smaller possible divisor.
2. Trial division only up to the square root.
3. The Sieve of Eratosthenes.

The expected results below act as small correctness checks.

In [ ]:
from time import perf_counter


def sum_primes_trial(limit: int) -> int:
    """Sum primes below limit by testing all smaller divisors."""
    prime_total = 0

    for candidate in range(2, limit):
        is_prime = True

        for divisor in range(2, candidate):
            if candidate % divisor == 0:
                is_prime = False
                break

        if is_prime:
            prime_total += candidate

    return prime_total


def sum_primes_sqrt(limit: int) -> int:
    """Sum primes below limit, testing divisors only through sqrt(candidate)."""
    prime_total = 0

    for candidate in range(2, limit):
        is_prime = True
        divisor = 2

        while divisor * divisor <= candidate:
            if candidate % divisor == 0:
                is_prime = False
                break
            divisor += 1

        if is_prime:
            prime_total += candidate

    return prime_total


def sum_primes_sieve(limit: int) -> int:
    """Use a sieve table to identify and sum all primes below limit."""
    if limit <= 2:
        return 0

    prime_flags = [True] * limit
    prime_flags[0:2] = [False, False]

    factor = 2
    while factor * factor < limit:
        if prime_flags[factor]:
            for composite in range(factor * factor, limit, factor):
                prime_flags[composite] = False
        factor += 1

    return sum(value for value, flag in enumerate(prime_flags) if flag)


In [ ]:
# Check that each approach gives the expected result on small inputs.
known_results = {10: 17, 30: 129, 100: 1060, 1000: 76127}

for method in (sum_primes_trial, sum_primes_sqrt, sum_primes_sieve):
    print(f"\\n{method.__name__}")
    for limit, expected in known_results.items():
        actual = method(limit)
        status = "PASS" if actual == expected else "CHECK"
        print(f"limit={limit:>4} | result={actual:>7} | {status}")


## Timing the first approach

The direct trial-division version becomes expensive as the limit grows. The following cell measures two input sizes. Actual timings depend on the computer running the notebook.

In [ ]:
for limit in (50_000, 100_000):
    started = perf_counter()
    result = sum_primes_trial(limit)
    elapsed = perf_counter() - started
    print(f"limit={limit:,} | sum={result:,} | elapsed={elapsed:.4f} seconds")


## Comparing the improved approaches

The square-root method avoids checking divisors beyond the square root because a composite number must have at least one factor no larger than that value. The sieve instead marks multiples of discovered primes, avoiding repeated primality tests.

In [ ]:
large_limit = 2_000_000

for method in (sum_primes_sqrt, sum_primes_sieve):
    started = perf_counter()
    result = method(large_limit)
    elapsed = perf_counter() - started
    print(f"{method.__name__}: sum={result:,}, time={elapsed:.4f} seconds")


## Observations

**Q3. What happened when N increased from 50,000 to 100,000?**

The basic method takes considerably longer because it checks many possible divisors for each candidate number. Increasing the limit increases the amount of repeated work.

**Q4. How did the square-root method compare with the basic method?**

It is faster because it only tests divisors up to the square root of each candidate. It still examines the numbers one at a time, however.

**Q5. Compare the methods for N = 2,000,000.**

The square-root method and sieve can both be timed in the comparison cell. The sieve is generally much faster for this larger limit. Exact timings vary by machine, so use the values printed by your own run.

**Q6. Which change helped the most?**

The sieve usually provides the largest improvement. It marks multiples of prime numbers as composite instead of independently testing every candidate.